# 02 — Exploratory Data Analysis

## Deep Learning in Asset Pricing — Replication Project

### Purpose of this notebook

This notebook examines the processed datasets created in `01_data_preparation.ipynb` before any asset-pricing model is estimated.

The analysis focuses on:

1. dataset dimensions and time coverage,
2. the number of stock observations available each month,
3. the distribution of monthly stock returns,
4. the distributions and cross-sectional behavior of the 46 firm characteristics,
5. the structure of the 178 macroeconomic predictors,
6. the overlap between the stock and macroeconomic datasets,
7. the chronological training, validation, and test samples.

> This notebook is diagnostic only. It does not change, normalize, winsorize, impute, or otherwise modify the processed replication data.

## 1. Exploratory Analysis Objectives

Before estimating linear, neural-network, LSTM, or adversarial asset-pricing models, it is important to verify the statistical properties of the data that will enter those models.

Exploratory analysis helps identify unexpected gaps, unusual return observations, changes in the cross-sectional number of stocks through time, characteristic distributions that differ from the expected rank-normalized scale, and differences between the stock and macroeconomic sample periods.

The stock-characteristic panel contains 46 firm characteristics and spans January 1967 through December 2016. The processed macroeconomic dataset contains 178 predictors and begins later, in January 1976.

Because the macro dataset starts later than the stock panel, models that combine firm characteristics with macroeconomic information will eventually use the common sample available in both datasets.

## 2. Imports and Reproducibility

This section loads the libraries required for exploratory analysis.

`pandas` and `numpy` are used for tabular and numerical analysis, while `matplotlib` is used for visualization.

A fixed random seed is used whenever we draw a subsample for computationally expensive diagnostics. The underlying processed datasets themselves are never randomly altered.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("NumPy  :", np.__version__)
print("pandas :", pd.__version__)
print("Seed   :", SEED)

## 3. Locate the Repository and Load the Processed Data

To keep the project reproducible across computers, this notebook does not use an absolute user-specific file path.

Instead, it searches upward for the project root and then loads the Parquet files created by the first notebook from `data/processed/`.

The two principal inputs are:

- `retchar_processed.parquet`
- `macro_processed.parquet`

Using the processed files ensures that this notebook analyzes exactly the same data that will later be used for model estimation.

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "requirements.txt").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open Jupyter from inside the cloned repository."
    )

ROOT = find_repo_root()
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root :", ROOT)
print("Processed data  :", PROCESSED_DIR)
print("Figures         :", FIGURES_DIR)
print("Tables          :", TABLES_DIR)

In [ ]:
RETCHAR_PATH = PROCESSED_DIR / "retchar_processed.parquet"
MACRO_PATH = PROCESSED_DIR / "macro_processed.parquet"

if not RETCHAR_PATH.exists():
    raise FileNotFoundError(f"{RETCHAR_PATH.name} not found. Run 01_data_preparation.ipynb first.")

if not MACRO_PATH.exists():
    raise FileNotFoundError(f"{MACRO_PATH.name} not found. Run 01_data_preparation.ipynb first.")

stock = pd.read_parquet(RETCHAR_PATH)
macro = pd.read_parquet(MACRO_PATH)

print(f"Stock data: {stock.shape[0]:,} rows × {stock.shape[1]:,} columns")
print(f"Macro data: {macro.shape[0]:,} rows × {macro.shape[1]:,} columns")

## 4. Identify the Data Structure

The processed stock file has one monthly date column, one return column, and 46 firm characteristics. The macro dataset contains one monthly date column and 178 macroeconomic predictors.

We identify these fields programmatically so the notebook remains robust to small naming differences.

In [ ]:
def detect_column(columns, candidates, label):
    cols = list(columns)
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(f"Could not detect {label}. Tried {candidates}. Available columns begin with {cols[:20]}")

DATE_COL = detect_column(stock.columns, ["date", "yyyymm", "month", "caldt", "time"], "stock date")
RET_COL = detect_column(stock.columns, ["ret", "return", "ret_excess", "excess_ret", "retx"], "return")
MACRO_DATE_COL = detect_column(macro.columns, ["date", "yyyymm", "month", "caldt", "time"], "macro date")

characteristic_cols = [
    c for c in stock.columns
    if c not in [DATE_COL, RET_COL] and pd.api.types.is_numeric_dtype(stock[c])
]

macro_feature_cols = [
    c for c in macro.columns
    if c != MACRO_DATE_COL and pd.api.types.is_numeric_dtype(macro[c])
]

print("Stock date column        :", DATE_COL)
print("Return column            :", RET_COL)
print("Firm characteristics     :", len(characteristic_cols))
print("Macro date column        :", MACRO_DATE_COL)
print("Macroeconomic predictors :", len(macro_feature_cols))

## 5. Dataset Overview and Time Coverage

For each processed dataset, we report the number of rows, columns, unique months, and the first and last month. This provides a compact verification of the empirical sample before examining individual variables.

In [ ]:
overview = pd.DataFrame([
    {
        "dataset": "Stock characteristics",
        "rows": len(stock),
        "columns": stock.shape[1],
        "months": stock[DATE_COL].nunique(),
        "start": stock[DATE_COL].min(),
        "end": stock[DATE_COL].max(),
    },
    {
        "dataset": "Macro",
        "rows": len(macro),
        "columns": macro.shape[1],
        "months": macro[MACRO_DATE_COL].nunique(),
        "start": macro[MACRO_DATE_COL].min(),
        "end": macro[MACRO_DATE_COL].max(),
    },
])

display(overview)

## 6. Cross-Sectional Number of Stock Observations Through Time

The processed stock dataset does not include a stock identifier, so individual firms cannot be tracked through time. However, each row is one stock-month observation.

Counting rows by month therefore measures the size of the stock cross-section available to the model. Large unexplained discontinuities would be important to investigate before estimation.

In [ ]:
monthly_stock_counts = (
    stock.groupby(DATE_COL)
    .size()
    .rename("stock_observations")
    .to_frame()
)

display(monthly_stock_counts.describe().T)

print("Minimum monthly observations:", f"{monthly_stock_counts['stock_observations'].min():,}")
print("Maximum monthly observations:", f"{monthly_stock_counts['stock_observations'].max():,}")
print("Median monthly observations :", f"{monthly_stock_counts['stock_observations'].median():,.0f}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_stock_counts.index, monthly_stock_counts["stock_observations"])
plt.title("Number of Stock Observations per Month")
plt.xlabel("Date")
plt.ylabel("Stock-month observations")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Monthly Stock Return Distribution

We examine the distribution of monthly stock returns to understand its center, dispersion, skewness, heavy tails, and extreme observations.

At this stage, extreme observations are documented rather than winsorized or removed. Any modification of returns would be an additional preprocessing choice and should not be introduced silently during exploratory analysis.

In [ ]:
return_summary = stock[RET_COL].describe(
    percentiles=[0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
)

display(return_summary.to_frame(name=RET_COL))

print("Return skewness:", f"{stock[RET_COL].skew():.4f}")
print("Return kurtosis:", f"{stock[RET_COL].kurt():.4f}")

In [ ]:
plot_n = min(250_000, len(stock))
return_plot_sample = stock[RET_COL].sample(n=plot_n, random_state=SEED)

q_low, q_high = return_plot_sample.quantile([0.005, 0.995])

plt.figure(figsize=(10, 5))
plt.hist(return_plot_sample.clip(q_low, q_high), bins=100)
plt.title("Distribution of Monthly Stock Returns (0.5%–99.5% clipped for display)")
plt.xlabel("Monthly return")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Clipping above is used only for visualization; the underlying data are unchanged.")

## 8. Returns Across Training, Validation, and Test Periods

We compare the return distribution across the fixed chronological samples:

- training: 1967–1986,
- validation: 1987–1991,
- test: 1992–2016.

This is descriptive only. Test-period statistics are not used to tune the models.

In [ ]:
TRAIN_START = pd.Timestamp("1967-01-01")
TRAIN_END = pd.Timestamp("1986-12-31")
VALID_START = pd.Timestamp("1987-01-01")
VALID_END = pd.Timestamp("1991-12-31")
TEST_START = pd.Timestamp("1992-01-01")
TEST_END = pd.Timestamp("2016-12-31")

def label_split(date):
    if TRAIN_START <= date <= TRAIN_END:
        return "train"
    if VALID_START <= date <= VALID_END:
        return "validation"
    if TEST_START <= date <= TEST_END:
        return "test"
    return "outside"

stock_eda = stock[[DATE_COL, RET_COL]].copy()
stock_eda["split"] = stock_eda[DATE_COL].map(label_split)

return_by_split = stock_eda.groupby("split")[RET_COL].agg(
    ["count", "mean", "std", "median", "min", "max"]
)

display(return_by_split.loc[
    [x for x in ["train", "validation", "test"] if x in return_by_split.index]
])

## 9. Firm-Characteristic Summary Statistics

The stock panel contains 46 firm characteristics. Because the supplied characteristics are already rank-normalized, their values should remain approximately within

\[
[-0.5, 0.5].
\]

For each characteristic, we calculate its mean, standard deviation, minimum, selected percentiles, and maximum.

In [ ]:
char_summary = stock[characteristic_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(char_summary)

print(
    "Overall characteristic range:",
    f"[{stock[characteristic_cols].min().min():.6f}, "
    f"{stock[characteristic_cols].max().max():.6f}]"
)

## 10. Selected Firm-Characteristic Distributions

We visualize a small set of representative characteristics. The notebook prefers commonly used variables when they are available and otherwise uses the first available characteristics.

Because the characteristics are rank-normalized cross-sectionally, many distributions should appear broadly spread across approximately \([-0.5,0.5]\).

In [ ]:
preferred_characteristics = ["a2me", "beme", "beta", "idiovol", "lev", "r12_2"]

selected_characteristics = [
    c for c in preferred_characteristics if c in characteristic_cols
]

if len(selected_characteristics) < 6:
    for c in characteristic_cols:
        if c not in selected_characteristics:
            selected_characteristics.append(c)
        if len(selected_characteristics) == 6:
            break

print("Selected characteristics:", selected_characteristics)

In [ ]:
plot_n = min(200_000, len(stock))
char_plot_sample = stock[selected_characteristics].sample(n=plot_n, random_state=SEED)

for c in selected_characteristics:
    plt.figure(figsize=(8, 4))
    plt.hist(char_plot_sample[c], bins=60)
    plt.title(f"Distribution of {c}")
    plt.xlabel(c)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 11. Cross-Sectional Characteristic Behavior Through Time

Rank normalization is performed within each month in the supplied processed data.

We summarize the monthly cross-sectional mean and standard deviation of each characteristic. Large changes would suggest unusual cross-sectional structure or differences in preprocessing.

In [ ]:
monthly_char_mean = stock.groupby(DATE_COL)[characteristic_cols].mean()
monthly_char_std = stock.groupby(DATE_COL)[characteristic_cols].std()

char_time_diagnostics = pd.DataFrame({
    "avg_abs_monthly_mean": monthly_char_mean.abs().mean(),
    "avg_monthly_std": monthly_char_std.mean(),
    "min_monthly_std": monthly_char_std.min(),
    "max_monthly_std": monthly_char_std.max(),
}).sort_values("avg_abs_monthly_mean", ascending=False)

display(char_time_diagnostics)

## 12. Correlation Among Firm Characteristics

The deep-learning models use many firm characteristics simultaneously. It is useful to understand the linear dependence among the inputs.

For efficiency, the correlation matrix is computed on a fixed random subsample. This subsample is used only for exploratory diagnostics; model estimation will use the intended full sample.

In [ ]:
corr_n = min(200_000, len(stock))
corr_sample = stock[characteristic_cols].sample(n=corr_n, random_state=SEED)

char_corr = corr_sample.corr()
display(char_corr.round(3))

abs_corr = char_corr.abs().copy()
np.fill_diagonal(abs_corr.values, np.nan)

stacked_corr = abs_corr.stack().sort_values(ascending=False)

print("Largest absolute pairwise characteristic correlations:")
display(stacked_corr.head(20).to_frame("absolute_correlation"))

## 13. Univariate Relation Between Characteristics and Returns

As a preliminary descriptive diagnostic, we compute simple correlations between each firm characteristic and contemporaneous returns.

These are not model estimates, causal effects, or trading strategies. They only indicate which characteristics have the strongest unconditional linear association with returns in the processed sample.

In [ ]:
return_corr_n = min(250_000, len(stock))

return_corr_sample = stock[[RET_COL] + characteristic_cols].sample(
    n=return_corr_n,
    random_state=SEED,
)

char_return_corr = (
    return_corr_sample[characteristic_cols]
    .corrwith(return_corr_sample[RET_COL])
    .sort_values(key=np.abs, ascending=False)
    .rename("correlation_with_return")
    .to_frame()
)

display(char_return_corr)

## 14. Macroeconomic Dataset Overview

The processed macroeconomic dataset contains 178 predictors and one observation per month.

We examine sample coverage and missingness before any recurrent-network model is estimated. The macro dataset begins in January 1976, later than the stock-characteristic panel.

In [ ]:
macro_quality = pd.Series({
    "rows": len(macro),
    "columns": macro.shape[1],
    "unique_months": macro[MACRO_DATE_COL].nunique(),
    "start": macro[MACRO_DATE_COL].min(),
    "end": macro[MACRO_DATE_COL].max(),
    "macro_predictors": len(macro_feature_cols),
    "total_missing_cells": int(macro[macro_feature_cols].isna().sum().sum()),
}, name="value")

display(macro_quality.to_frame())

In [ ]:
macro_missing = (
    macro[macro_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .to_frame()
)

display(macro_missing.head(25))

## 15. Macroeconomic Predictor Summary Statistics

Different macroeconomic variables can have very different economic meanings and scales.

This section reports summary statistics for all numeric macro predictors. No additional standardization is performed here; the objective is to document the supplied processed inputs.

In [ ]:
macro_summary = macro[macro_feature_cols].describe().T
display(macro_summary)

## 16. Selected Macroeconomic Time Series

We visualize a small number of macroeconomic predictors through time to identify discontinuities, changing volatility, unusual outliers, or possible nonstationary behavior.

The first four numeric macro predictors are selected automatically so the notebook remains compatible with the supplied file.

In [ ]:
selected_macro = macro_feature_cols[:4]

print("Selected macro predictors:", selected_macro)

for c in selected_macro:
    plt.figure(figsize=(11, 4))
    plt.plot(macro[MACRO_DATE_COL], macro[c])
    plt.title(f"Macroeconomic Predictor: {c}")
    plt.xlabel("Date")
    plt.ylabel(c)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 17. Stock–Macro Time Alignment

The stock-characteristic panel covers January 1967 through December 2016, while the macroeconomic dataset begins in January 1976.

We compare their monthly date sets explicitly. Models using only firm characteristics can potentially use the full stock sample, while models requiring macroeconomic state variables can use only the common period unless an alternative treatment is justified.

In [ ]:
stock_months = set(stock[DATE_COL].dropna().unique())
macro_months = set(macro[MACRO_DATE_COL].dropna().unique())

shared_months = stock_months & macro_months
stock_only_months = stock_months - macro_months
macro_only_months = macro_months - stock_months

alignment_summary = pd.Series({
    "stock_months": len(stock_months),
    "macro_months": len(macro_months),
    "shared_months": len(shared_months),
    "stock_only_months": len(stock_only_months),
    "macro_only_months": len(macro_only_months),
}, name="months")

display(alignment_summary.to_frame())

if shared_months:
    print("Common sample starts:", min(shared_months))
    print("Common sample ends  :", max(shared_months))

## 18. Verify the Saved Train / Validation / Test Files

The first notebook saved the chronological samples separately.

This section verifies that those files are available and reports their dimensions so later modeling notebooks can load the fixed samples directly.

In [ ]:
split_files = {
    "stock_train": "retchar_train.parquet",
    "stock_validation": "retchar_validation.parquet",
    "stock_test": "retchar_test.parquet",
    "macro_train": "macro_train.parquet",
    "macro_validation": "macro_validation.parquet",
    "macro_test": "macro_test.parquet",
}

split_inventory = []

for label, filename in split_files.items():
    path = PROCESSED_DIR / filename

    if path.exists():
        df_tmp = pd.read_parquet(path)
        date_col = DATE_COL if label.startswith("stock") else MACRO_DATE_COL

        split_inventory.append({
            "dataset": label,
            "rows": len(df_tmp),
            "columns": df_tmp.shape[1],
            "months": df_tmp[date_col].nunique(),
            "start": df_tmp[date_col].min(),
            "end": df_tmp[date_col].max(),
        })
    else:
        split_inventory.append({
            "dataset": label,
            "rows": np.nan,
            "columns": np.nan,
            "months": np.nan,
            "start": pd.NaT,
            "end": pd.NaT,
        })

split_inventory = pd.DataFrame(split_inventory)
display(split_inventory)

## 19. Save Exploratory Summary Tables

The main exploratory results are saved in `results/tables/` so they can later be used in the replication report and compared with the paper.

In [ ]:
overview.to_csv(TABLES_DIR / "eda_dataset_overview.csv", index=False)
monthly_stock_counts.to_csv(TABLES_DIR / "eda_monthly_stock_counts.csv")
char_summary.to_csv(TABLES_DIR / "eda_characteristic_summary.csv")
char_return_corr.to_csv(TABLES_DIR / "eda_characteristic_return_correlations.csv")
macro_missing.to_csv(TABLES_DIR / "eda_macro_missingness.csv")
macro_summary.to_csv(TABLES_DIR / "eda_macro_summary.csv")
split_inventory.to_csv(TABLES_DIR / "eda_split_inventory.csv", index=False)

print("✓ Exploratory summary tables saved to:")
print(TABLES_DIR)

## 20. Exploratory Analysis Checklist

Before moving to the benchmark model, verify:

- [ ] The stock panel has 46 firm characteristics.
- [ ] The stock sample spans January 1967 through December 2016.
- [ ] The monthly stock cross-section has been inspected.
- [ ] The return distribution and extreme observations have been inspected.
- [ ] Firm characteristics remain approximately within the expected rank-normalized range.
- [ ] Characteristic correlations have been inspected.
- [ ] The macro dataset contains 178 numeric predictors.
- [ ] Macro missingness has been documented.
- [ ] The macro sample begins in January 1976.
- [ ] The stock–macro common period has been verified.
- [ ] Training, validation, and test files have been confirmed.

## Next Notebook

**`03_linear_baseline.ipynb`**

The next stage should establish a transparent benchmark before introducing deep learning. The linear model will provide a reference point for evaluating whether the feedforward, LSTM, and adversarial architectures improve asset-pricing performance.